In [1]:
#密度vs半径グラフ（jeans_3d-test14を可視化するときに使った）
#レーンエムデン用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")

# 球・計算領域
xc, yc, zc = 6.0, 6.0, 6.0   # 球中心
r_max = 6.0                 # 球半径（Rsphere=5 だが少し余裕）
n_bins = 200                # 球殻数（滑らかさはここで決まる）

# 出力
output_dir = "./radial_density_profiles"
os.makedirs(output_dir, exist_ok=True)

# ======================================
# VTK ファイルを timestep ごとに整理
# ======================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

step_dict = {}
for f in vtk_files:
    step = f.split('.')[-2]   # 00000 など
    step_dict.setdefault(step, []).append(f)

print(f"Found {len(step_dict)} timesteps")

# ======================================
# 球殻ビン定義（r=0 を必ず含む）
# ======================================
bin_edges = np.linspace(0.0, r_max, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# ======================================
# 各 timestep の処理
# ======================================
for step, files in step_dict.items():

    # --- 全 block を結合 ---
    blocks = [pv.read(f) for f in files]
    grid = pv.MultiBlock(blocks).combine()

    print(f"[step {step}] bounds = {grid.bounds}")

    # --- セル中心と密度 ---
    centers = grid.cell_centers().points
    rho = grid["dens"]

    # --- 半径 ---
    dx = centers[:, 0] - xc
    dy = centers[:, 1] - yc
    dz = centers[:, 2] - zc
    r = np.sqrt(dx*dx + dy*dy + dz*dz)

    # --- 有効領域のみ ---
    valid = np.isfinite(rho) & (rho > 0) & (r <= r_max)
    r = r[valid]
    rho = rho[valid]

    # ======================================
    # 球殻平均
    # ======================================
    rho_shell = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)

    bin_index = np.digitize(r, bin_edges) - 1

    for i in range(n_bins):
        mask = bin_index == i
        if np.any(mask):
            rho_shell[i] = rho[mask].mean()
            counts[i] = np.count_nonzero(mask)
        else:
            rho_shell[i] = np.nan

    # --- 中心ビンの確認 ---
    print(f"  center bin: r~{bin_centers[0]:.3e}, "
          f"rho={rho_shell[0]:.6f}, N={counts[0]}")

    # ======================================
    # プロット
    # ======================================
    plt.figure(figsize=(6, 4))
    plt.plot(bin_centers, rho_shell, lw=2, color="C0")
    plt.scatter(bin_centers, rho_shell, s=10, color="C0")

    plt.xlabel("r")
    plt.ylabel("Density")
    plt.xlim(0, 2.0)
    plt.ylim(-0.1,1.0)
    plt.title(f"Spherically averaged density (step {step})")
    plt.grid(True)

    png = os.path.join(output_dir, f"radial_density_shell_{step}.png")
    plt.savefig(png, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Saved {png}")

print("Done.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/aian/athenapp/results/〇〇'

In [ ]:
#半径vs密度　縦軸logスケール（jeans_3d-test14を可視化するときに使った）
#レーンエムデン用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
import matplotlib.animation as animation
from PIL import Image

vtk_dir = os.path.expanduser("~/athenapp/results/jeans_3d-test27")

# 球・計算領域
xc, yc, zc = 6.0, 6.0, 6.0   # 球中心
r_max = 6.0                 # 球半径（Rsphere=5 だが少し余裕）
n_bins = 200                # 球殻数（滑らかさはここで決まる）

# 出力
output_dir = "./radial_density_profiles_log"
os.makedirs(output_dir, exist_ok=True)

# ======================================
# VTK ファイルを timestep ごとに整理
# ======================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

step_dict = {}
for f in vtk_files:
    step = f.split('.')[-2]   # 00000 など
    step_dict.setdefault(step, []).append(f)

print(f"Found {len(step_dict)} timesteps")

# ======================================
# 球殻ビン定義（r=0 を必ず含む）
# ======================================
bin_edges = np.linspace(0.0, r_max, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# ======================================
# 各 timestep の処理
# ======================================
for step, files in step_dict.items():

    # --- 全 block を結合 ---
    blocks = [pv.read(f) for f in files]
    grid = pv.MultiBlock(blocks).combine()

    print(f"[step {step}] bounds = {grid.bounds}")

    # --- セル中心と密度 ---
    centers = grid.cell_centers().points
    rho = grid["dens"]

    # --- 半径 ---
    dx = centers[:, 0] - xc
    dy = centers[:, 1] - yc
    dz = centers[:, 2] - zc
    r = np.sqrt(dx*dx + dy*dy + dz*dz)

    # --- 有効領域のみ ---
    valid = np.isfinite(rho) & (rho > 0) & (r <= r_max)
    r = r[valid]
    rho = rho[valid]

    # ======================================
    # 球殻平均
    # ======================================
    rho_shell = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)

    bin_index = np.digitize(r, bin_edges) - 1

    for i in range(n_bins):
        mask = bin_index == i
        if np.any(mask):
            rho_shell[i] = rho[mask].mean()
            counts[i] = np.count_nonzero(mask)
        else:
            rho_shell[i] = np.nan

    # --- 中心ビンの確認 ---
    print(f"  center bin: r~{bin_centers[0]:.3e}, "
          f"rho={rho_shell[0]:.6f}, N={counts[0]}")

    # ======================================
    # プロット
    # ======================================
    rho_floor = 1e-8   # ← 見たい精度に応じて調整（1e-8 などでもOK）

    rho_plot = rho_shell.copy()
    rho_plot[(rho_plot <= rho_floor) | ~np.isfinite(rho_plot)] = np.nan

    plt.figure(figsize=(6, 4))
    plt.plot(bin_centers, rho_plot, lw=0.6, color="C0")
    plt.scatter(bin_centers, rho_plot, s=3, color="C0", alpha=0.4)

    plt.xlabel("r")
    plt.ylabel("Density")
    plt.xlim(0, 6.0)
    plt.yscale("log")
    plt.ylim(rho_floor, 1.5)

    plt.title(f"Spherically averaged density (File {step})")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    
    png = os.path.join(output_dir, f"radial_density_log_{step}.png")
    plt.savefig(png, dpi=150, bbox_inches="tight")
    plt.close()
    
print("Done.")

In [ ]:
#Toyouchi-test4を解析するときに使った

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort

vtk_dir = os.path.expanduser("~/athenapp/results/Toyouchi-test4")

# ===============================
# 球設定（今回の計算に合わせる）
# ===============================
xc, yc, zc = 0.0, 0.0, 0.0   # ← 中心
r_max = 200.0                 # ← 計算領域の半分より少し小さく
n_bins = 20

# 出力
output_dir = "./radial_density_profiles_toyouchi"
os.makedirs(output_dir, exist_ok=True)

# ======================================
# VTK ファイル整理
# ======================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

step_dict = {}
for f in vtk_files:
    step = f.split('.')[-2]
    step_dict.setdefault(step, []).append(f)

print(f"Found {len(step_dict)} timesteps")

# ======================================
# 球殻ビン
# ======================================
bin_edges = np.logspace(np.log10(1.5), np.log10(r_max), n_bins + 1)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

# ======================================
# Toyouchi 理論分布
# ======================================
def toyouchi_profile(r):
    return r**(-1.75)

# ======================================
# timestep ループ
# ======================================
for step, files in step_dict.items():

    blocks = [pv.read(f) for f in files]
    grid = pv.MultiBlock(blocks).combine()

    print(f"[step {step}] bounds = {grid.bounds}")

    centers = grid.cell_centers().points
    rho = grid["dens"]

    # 半径
    dx = centers[:,0] - xc
    dy = centers[:,1] - yc
    dz = centers[:,2] - zc
    r = np.sqrt(dx*dx + dy*dy + dz*dz)

    valid = np.isfinite(rho) & (rho > 0) & (r <= r_max)

    r = r[valid]
    rho = rho[valid]

    # ======================================
    # 球殻平均
    # ======================================
    rho_shell = np.zeros(n_bins)

    bin_index = np.digitize(r, bin_edges) - 1

    for i in range(n_bins):

        mask = bin_index == i

        if np.any(mask):
            rho_shell[i] = rho[mask].mean()
        else:
            rho_shell[i] = np.nan

    # ======================================
    # 理論分布（正規化）
    # ======================================
    theory = toyouchi_profile(bin_centers)

    # スケール合わせ
    scale = np.nanmedian(rho_shell / theory)
    theory *= scale

    # ======================================
    # プロット
    # ======================================
    plt.figure(figsize=(6,4))

    plt.plot(bin_centers, rho_shell,
             lw=1.0,
             label="Simulation")

    plt.plot(bin_centers, theory,
             "--",
             lw=0.5,
             label=r"$\rho \propto r^{-1.75}$")

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel("r")
    plt.ylabel("Density")

    plt.title(f"Spherical density profile (step {step})")

    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()

    png = os.path.join(output_dir,
                       f"density_profile_{step}.png")

    plt.savefig(png, dpi=150, bbox_inches="tight")
    plt.close()

    
print("min r =", np.min(r))
print("max r =", np.max(r))
print("Done.")